# Maze Solver Implementation
## Maze Search Algorithms Assignment 

### Table of Contents 
1. Project Overiew 
2. Maze Parsing and I/O
3. Shared Utilitie s
4. Blind Search Algorithms 
5. Heuristic Search Algorithms 
6. Interface Functions 
7. Testing and Examples
8. Performance and Analysis 

## Project Overview 
This implementation will include 3 search algorithms for the maze solving: 
2 Blind Search Algorithms:
- **Breadth-First Search (BFS)**: this will find and guarantee the shortest path from Start to Goal
- **Deep-First Search (DFS)**: this will go the deepest path from Start to Goal, and retrace if needed; minimial memory
- **Greedy Best-First Search**: Heuristic search, picks the best looking path

### Maze Format
'S' - Start position
'E' - End position
'*' - Path taken / solution
'X' - Wall
' ' - Open Space 

- using a text file format (.txt) for output and dimensions of maze labels on the first line
- 
    

## Maze Parsing and I/O
- These functions will handle the reading of the maze files and converting them to the correct data scructure

In [89]:
from collections import deque # used for BFS and DFS operations
import math # heuristic calulcations 


def get_maze(maze_text):

    #reads in a multi-lined string and splits them and stores them in a list. Lines are in reference to match dimensions inputted by the user, and strip any whitespace
    lines = maze_text.strip().split("\n")

    # read the first line which contains the suggested dimensions of the maze
    # unpack the width and height from dimensions
    dimensions = lines[0].split()
    width, height = int(dimensions[0]), int(dimensions[1])

    grid = []
    start_position = None
    end_position = None 

    #formatting the maze using the width and height 
    # going to skip the first line cause it displays the dimensions
    for row_index, line in enumerate(lines[1:]): 
        # set width and pad if needed for rows, then append
        row = list(line.ljust(width))
        grid.append(row)

        # set column, iterate through the rows to format and find Start and End positions 
        for column_index, cells in enumerate(row):
            if cells == 'S':
                start_position = (row_index, column_index)
            elif cells == 'E':
                end_position = (row_index, column_index)

    # return the new grid, start/end position, and width/height
    return {
        "width": width,
        "height": height,
        "grid": grid,
        "start": start_position,
        "goal": end_position
    }
            
def get_grid(maze_data):
 # display the grid utilizing the parsed maze data 
    print(f'{maze_data['width']} (width) x {maze_data['height']} (height)')
    
    for row in maze_data['grid']:
        print(''.join(row))
    

In [90]:
test_maze = """10 6
XXXXXXXXXX
X        S
X XXXXXX X
X X    XXX
X   XX   E
XXXXXXXXXX"""



test_maze2 = """8 8

XXXXXXXX

X      X

X  X   X

E  X   X

X  X   S

X  XX  X

X      X

XXXXXXXX"""

In [91]:
maze_data = get_maze(test_maze)
print("Parsed maze data:")
print(f"Dimensions: {maze_data['width']} x {maze_data['height']}")
print(f"Start position: {maze_data['start']}")
print(f"End position: {maze_data['goal']}")

get_grid(maze_data)


Parsed maze data:
Dimensions: 10 x 6
Start position: (1, 9)
End position: (4, 9)
10 (width) x 6 (height)
XXXXXXXXXX
X        S
X XXXXXX X
X X    XXX
X   XX   E
XXXXXXXXXX


save for later ''''def read_maze_file(filepath):
    """Read maze from file and parse it"""
    with open(filepath, 'r') as file:
        return get_maze(file.read())

 ## 3. Shared Utilities 

Helper functions like classes that will be used by all the algorithms to help avoid duplicate / extensive code and maintain consistency 

It will check for Position Validation, Neighbor Discovery( adjacent valid positions in the current location of the maze), and path recognition specific to the search algorithm's data

this will build the foundation in which all 3 search algorithms will build upon

In [136]:
def is_position_valid(maze_data, row, col):
    # checking for is a position is valid, boundaries of the walls 

    # check row boundary
    if row < 0 or row >= maze_data['height']: return False

    #check column boundary
    if col < 0 or col >= maze_data['width'] : return False

    # checking if the position is a wall or not 
    cell = maze_data['grid'][row][col]
    if cell == 'X':
        return False

    # otherwise, return true
    return True

In [137]:
# test the function 
print(f"Position(0,0): {is_position_valid(maze_data, 0, 0)}")
print(f"Position(1,1): {is_position_valid(maze_data, 1, 1)}")
print(f"Position(-1,1): {is_position_valid(maze_data, -1, 1)}")

Position(0,0): False
Position(1,1): True
Position(-1,1): False


In [138]:
def get_neighboring_positions(maze_data, position):
    #unpack to get current position
    row, col = position

    # store the valid neighbors in a empty list 
    neighbors = []

    # possible moves: up, down , left, right
    moves = [(row - 1, col), (row + 1, col), (row, col - 1), (row, col + 1)]
    # check possible moves, then append
    for new_row, new_col in moves:
        if is_position_valid(maze_data, new_row, new_col):
            neighbors.append((new_row, new_col))

    return neighbors 
    

In [139]:
# test get_neighboring_positions function
test_position = (1, 1)
print(get_neighboring_positions(maze_data, test_position))

[(2, 1), (1, 2)]


### Path Reconstruction
The final utility that rebuilds the complete solution path based off the desired such algorithm.
Utilizing the builtin features from the the search algorithms, it will be able recall how each position was reached and retrace our steps back through from end to start.



In [140]:
def construct_path(came_from, start, goal):
    # creating an empty list, to have somewhere to store the path that was built
    path = []
    current_position = goal # working backwards from goal to start. each position will point to where i came from

    # as we work backwards from goal to start, it helps us keep track of where we came_from rather than trying to figure out how to get to.
    # seems like a lot easier impementation 
    # until 'start' is reached, the while loop will continue to run
    while current_position != start:
        # add current_position to empty list, basically logging each movement (from nodes to edges)
        path.append(current_position)
        current_position = came_from[current_position]

    # once, start is acheived, add to the 'path' list. Its technially not in came_from because we are starting from 'goal' working toward 'start'
    path.append(start)
    path.reverse()

    return path
    

In [141]:
# testing construct_path function
sample_came_from = {
    (1,2): (1,1),
    (2,2): (1,2),
    (2,3): (2,2),
    (2,4): (2,3)
}

start = (1,1)
goal = (2,4)

path = construct_path(sample_came_from, start, goal)
print(f'constructed path: {path}')

constructed path: [(1, 1), (1, 2), (2, 2), (2, 3), (2, 4)]


## 4. Blind Search Algorithms
Blind-search algorithms are type of algorithm where the initial starting state/point and end goal are unknown. This can also be referred to as uninformed search where a predefined method is followed even with little to no knowledge to the goal state. 

- I will be using the Breadth-Search Algorithm (BFS) and Depth-Search Algorithm (DFS)

### Breadth-First Search
- attempting to find the shortest path, exploring the number of edges by priortizing fastest and simpliest paths. Which will be used to solve the maze quickly. It will require more memory depending on the size of the maze

### Dept-First Search
- This algorithm method will explore as deep into the maze before it will need to backtrack and correct the path to reach the goal. It will use less memory but take longer paths to ultimatly get from start to goal

In [142]:
from collections import deque

def breadth_first_search(maze_data):
    # retrieveing start and goal positions from maze_data
    start = maze_data['start']
    goal = maze_data['goal']


    # BFS is queue based, need to create a queue and use the collections library
 ver
                

    # if not path found, just return the most current
    return []
    







In [144]:
# testing BFS 
print('BFS Test:')
bfs_path = breadth_first_search(maze_data)
print(bfs_path)

print(f"width={maze_data['width']}, height={maze_data['height']}")
print(maze_data['start'])
print(maze_data['goal'])
print(is_position_valid(maze_data, maze_data['start'][0], maze_data['start'][1]))
print(is_position_valid(maze_data, maze_data['goal'][0], maze_data['goal'][1]))

start_row, start_col = maze_data['start']
print(f"row {start_row} < {maze_data['height']}, col {start_col}")

print(f"Cell at start (1,9): '{maze_data['grid'][1][9]}'")
print(f"Cell at goal (4,9): '{maze_data['grid'][4][9]}'")
print(f"Length of row 1: {len(maze_data['grid'][1])}")
print(f"Length of row 4: {len(maze_data['grid'][4])}")


print(f"Testing (1,1): {is_position_valid(maze_data, 1, 1)}")  # Should be True

BFS Test:
[(1, 9), (1, 8), (1, 7), (1, 6), (1, 5), (1, 4), (1, 3), (1, 2), (1, 1), (2, 1), (3, 1), (4, 1), (4, 2), (4, 3), (3, 3), (3, 4), (3, 5), (3, 6), (4, 6), (4, 7), (4, 8), (4, 9)]
width=10, height=6
(1, 9)
(4, 9)
True
True
row 1 < 6, col 9
Cell at start (1,9): 'S'
Cell at goal (4,9): 'E'
Length of row 1: 10
Length of row 4: 10
Testing (1,1): True


### Depth-First Search

Explores as far as possible along each edge hopefully reaching the goal, if not it will backtrack and retest another route
- uses less memory
- doesn't guarantee the shortest path due to the algorithm having to backtrack, in comparison to BFS which uses (FIFO - first in, first out) to get to the goal the quickest
- DFS uses the opposite of FIFO, since we continue on the path as far as we can, essentially we are just checking off where we visited and moving to the next space. If this first path of nodes and edges reaches the goal then we succeeded, if not and we come to a deadend, we back track and try another path. In this process we will us LIFO - Last in, First Out. Basically removing from the end

In [157]:
def depth_first_search(maze_data):
    # since we focus on exploring the farthest path possible before backtracking, this function will focus on stacking instead of queueing like BFS
    
    # retrieve the start and goal
    start = maze_data['start']
    goal = maze_data['goal']

    #stack and pop after visited
    stack = [start]
    visited = set([start])

    # keep track of where we visited, creating a dictionary of lists
    came_from = {}

    while stack:
        current_position = stack.pop()

        if current_position == goal:
            return construct_path(came_from, start, goal)

        for neighboring_cell in get_neighboring_positions(maze_data, current_position):
            if neighboring_cell not in visited: 
                visited.add(neighboring_cell)
                came_from[neighboring_cell] = current_position
                stack.append(neighboring_cell)

    # if no path not found return an empty list
    return []
    

In [158]:
print("Testing DSF:")
dfs_path = depth_first_search(maze_data)
print(dfs_path)
print(f'DFS path length {len(dfs_path)}')

Testing DSF:
[(1, 9), (1, 8), (1, 7), (1, 6), (1, 5), (1, 4), (1, 3), (1, 2), (1, 1), (2, 1), (3, 1), (4, 1), (4, 2), (4, 3), (3, 3), (3, 4), (3, 5), (3, 6), (4, 6), (4, 7), (4, 8), (4, 9)]
DFS path length 22


In [160]:
print("BFS path:", bfs_path, "\n", len(bfs_path))
print("DFS path:", bfs_path, "\n", len(bfs_path))
print("Paths are identical", bfs_path == dfs_path)

BFS path: [(1, 9), (1, 8), (1, 7), (1, 6), (1, 5), (1, 4), (1, 3), (1, 2), (1, 1), (2, 1), (3, 1), (4, 1), (4, 2), (4, 3), (3, 3), (3, 4), (3, 5), (3, 6), (4, 6), (4, 7), (4, 8), (4, 9)] 
 22
DFS path: [(1, 9), (1, 8), (1, 7), (1, 6), (1, 5), (1, 4), (1, 3), (1, 2), (1, 1), (2, 1), (3, 1), (4, 1), (4, 2), (4, 3), (3, 3), (3, 4), (3, 5), (3, 6), (4, 6), (4, 7), (4, 8), (4, 9)] 
 22
Paths are identical True


In [161]:
print(f"Start: {maze_data['start']}")  
print(f"Goal: {maze_data['goal']}")
print(f"Character at start: '{maze_data['grid'][maze_data['start'][0]][maze_data['start'][1]]}'")
print(f"Character at goal: '{maze_data['grid'][maze_data['goal'][0]][maze_data['goal'][1]]}'")

Start: (1, 9)
Goal: (4, 9)
Character at start: 'S'
Character at goal: 'E'


In [162]:
# Check the cells directly between start and goal
start_row, start_col = maze_data['start'] 
goal_row, goal_col = maze_data['goal']

print("Checking vertical path between start and goal:")
for row in range(start_row, goal_row + 1):
    cell = maze_data['grid'][row][start_col]  # Same column (9)
    print(f"Row {row}, Col {start_col}: '{cell}'")

Checking vertical path between start and goal:
Row 1, Col 9: 'S'
Row 2, Col 9: 'X'
Row 3, Col 9: 'X'
Row 4, Col 9: 'E'


In [164]:
# Check what's actually in the direct path
start_row, start_col = maze_data['start'] 
goal_row, goal_col = maze_data['goal']

print("Checking direct vertical path:")
for row in range(start_row, goal_row + 1):
    cell = maze_data['grid'][row][start_col]  # Column 9
    print(f"Row {row}, Col {start_col}: '{cell}'")

print("Neighbors of start:")
print(get_neighboring_positions(maze_data, maze_data['start']))    

Checking direct vertical path:
Row 1, Col 9: 'S'
Row 2, Col 9: 'X'
Row 3, Col 9: 'X'
Row 4, Col 9: 'E'
Neighbors of start:
[(1, 8)]


In [165]:
def display_maze_with_coordinates(maze_data):
    """
    Display the maze with coordinate labels to see exactly what's in each cell
    """
    print(f"Maze: {maze_data['width']} x {maze_data['height']}")
    print(f"Start: {maze_data['start']}, Goal: {maze_data['goal']}")
    print()
    
    # Print column headers
    print("   ", end="")
    for col in range(maze_data['width']):
        print(f" {col}", end="")
    print()
    
    # Print each row with row number and cell contents
    for row in range(maze_data['height']):
        print(f"{row}: ", end="")
        for col in range(maze_data['width']):
            cell = maze_data['grid'][row][col]
            print(f" {cell}", end="")
        print()  # New line after each row

# Test the visual display
display_maze_with_coordinates(maze_data)

Maze: 10 x 6
Start: (1, 9), Goal: (4, 9)

    0 1 2 3 4 5 6 7 8 9
0:  X X X X X X X X X X
1:  X                 S
2:  X   X X X X X X   X
3:  X   X         X X X
4:  X       X X       E
5:  X X X X X X X X X X


### Greedy Best-First Search
A heuristic function based algorithm that will make optimal choices for each step to reach the goal. This algorithm already has specific knowledge of the start and goal positions.

- decide to use Manhattan distance heuristic function because it is commonly used for grid-based movements, such like a maze (formatted in a grid-like shape and movement)
- this will help develop the Greedy best first search function

In [171]:
def manhattan_distance(pos1, pos2):
    
    row1, col1 = pos1
    row2, col2 = pos2 

    distance = abs(row1-row2) + abs(col1-col2)
    return distance

In [172]:
import math 
# testing the Manhattan Distance function 
# Test the Manhattan distance function
start = maze_data['start']  # (1, 9)
goal = maze_data['goal']    # (4, 9)

print(f"Manhattan distance from start {start} to goal {goal}: {manhattan_distance(start, goal)}")
print(f"Manhattan distance from (1,1) to goal: {manhattan_distance((1,1), goal)}")

Manhattan distance from start (1, 9) to goal (4, 9): 3
Manhattan distance from (1,1) to goal: 11


In [178]:
import heapq # using heapq to create priority queue which enables to the function to find the best option next. Usually the lowest value

# for reference
# https://docs.python.org/3/library/heapq.html#module-heapq

def greedy_best_first_search(maze_data):

    # getting the start and goal data
    start = maze_data['start']
    goal = maze_data['goal']

    # checking for priority with heapq
    # lowest explored first
    # store in empty list, to keep track
    priority_queue = []
    heapq.heappush(priority_queue, (manhattan_distance(start, goal), start))
    '''
    manhattan to give us the value of the shortest distance and  we assign that to start, 
    because thats we essentially need to come from to get to
    '''

    # log where we visited and came_from
    visited = set([start])
    came_from = {} 

    while priority_queue: 
        current_distance, current = heapq.heappop(priority_queue) # get positon with lowest heuristic distance

        if current == goal:
            return construct_path(came_from, start, goal)

        for neighboring_cell in get_neighboring_positions(maze_data, current):
            if neighboring_cell not in visited: 
                visited.add(neighboring_cell)
                came_from[neighboring_cell] = current
            
                # calculate the hueristic distance for the neighbor
                # using the Manhattan distance function 
                heuristic_distance = manhattan_distance(neighboring_cell, goal)
                heapq.heappush(priority_queue, (heuristic_distance, neighboring_cell))

    # if not found return empty string
    return []
        
        




In [179]:
print("Testing Greedy Best-First Search:")
gbf_path = greedy_best_first_search(maze_data)
print(f"GBF path: {gbf_path}")
print(f"GBF path length: {len(gbf_path)}")

Testing Greedy Best-First Search:
GBF path: [(1, 9), (1, 8), (1, 7), (1, 6), (1, 5), (1, 4), (1, 3), (1, 2), (1, 1), (2, 1), (3, 1), (4, 1), (4, 2), (4, 3), (3, 3), (3, 4), (3, 5), (3, 6), (4, 6), (4, 7), (4, 8), (4, 9)]
GBF path length: 22


In [180]:
print("Comparing all three algorithms:")
print(f"BFS path length: {len(bfs_path)}")
print(f"DFS path length: {len(dfs_path)}") 
print(f"GBF path length: {len(gbf_path)}")
print(f"All paths reach goal: BFS={bfs_path[-1] == maze_data['goal']}, DFS={dfs_path[-1] == maze_data['goal']}, GBF={gbf_path[-1] == maze_data['goal']}")

Comparing all three algorithms:
BFS path length: 22
DFS path length: 22
GBF path length: 22
All paths reach goal: BFS=True, DFS=True, GBF=True


In [ ]:
## 6. Interface Functions
calling and testing the 3 designed search algorithms 
2 Blind Search
- BFS & DFS 
1 Heuristic search
- Greedy Best-First Search


In [186]:
# function will be used to mark the path the algorithm took to get from "start" to "goal"

def mark_solution_path(maze_data, path):
    solved_grid = [row.copy() for row in maze_data["grid"]]

    for position in path:
        row, col = position # get current position 

        # using '*' to marked a visited position, verify that its not the start or goal position
        if solved_grid[row][col] not in ["S", "E"]:
            solved_grid[row][col] = "*"

    final_maze_output = f'{maze_data['width']} (width) x {maze_data['height']} (height)\n'
    for row in solved_grid:
        final_maze_output += ''.join(row) + "\n"


    # return final maze output with marked path
    return final_maze_output
        
        

In [188]:
test_path = bfs_path

solution = mark_solution_path(maze_data, test_path)

print(test_maze + "\n")

print(solution)

10 6
XXXXXXXXXX
X        S
X XXXXXX X
X X    XXX
X   XX   E
XXXXXXXXXX

10 (width) x 6 (height)
XXXXXXXXXX
X********S
X*XXXXXX X
X*X****XXX
X***XX***E
XXXXXXXXXX



In [185]:
''' solves for BFS'''
def maze_solver_one(maze):
    # will use maze solver one to solve BFS 
    # retrieve the maze data from the get_maze function
    maze_data = get_maze(maze)
    bfs_path = breadth_first_search(maze_data)

    # return the solution 
    if bfs_path:
        return mark_solution_path(maze_data, bfs_path)
    else:
        return maze
        
''' solves for DFS'''
def maze_solver_two(maze):
    # will use maze solver two to solve DFS
    # retrieve the maze data from the get_maze function
    maze_data = get_maze(maze)
    dfs_path = depth_first_search(maze_data)

    # return the solution 
    if dfs_path:
        return mark_solution_path(maze_data, dfs_path)
    else:
        return maze

'''solves for Greedy best-first-search'''
def maze_solver_three(maze):
    # will use maze solver two to solve gbfs
    # retrieve the maze data from the get_maze function
    maze_data = get_maze(maze)
    gbfs_path = greedy_best_first_search(maze_data)

    # return the solution 
    if gbfs_path:
        return mark_solution_path(maze_data, gbfs_path)
    else:
        return maze

                        

In [189]:
result = maze_solver_one(test_maze)
print(result)

10 (width) x 6 (height)
XXXXXXXXXX
X********S
X*XXXXXX X
X*X****XXX
X***XX***E
XXXXXXXXXX



In [194]:
%%writefile maze_solvers.py



from collections import deque # used for BFS and DFS operations
import heapq
import math # heuristic calulcations 


'''-----------GET MAZE & GRID DATA START-----------'''

def get_maze(maze_text):

    #reads in a multi-lined string and splits them and stores them in a list. Lines are in reference to match dimensions inputted by the user, and strip any whitespace
    lines = maze_text.strip().split("\n")

    # read the first line which contains the suggested dimensions of the maze
    # unpack the width and height from dimensions
    dimensions = lines[0].split()
    width, height = int(dimensions[0]), int(dimensions[1])

    grid = []
    start_position = None
    end_position = None 

    #formatting the maze using the width and height 
    # going to skip the first line cause it displays the dimensions
    for row_index, line in enumerate(lines[1:]): 
        # set width and pad if needed for rows, then append
        row = list(line.ljust(width))
        grid.append(row)

        # set column, iterate through the rows to format and find Start and End positions 
        for column_index, cells in enumerate(row):
            if cells == 'S':
                start_position = (row_index, column_index)
            elif cells == 'E':
                end_position = (row_index, column_index)

    # return the new grid, start/end position, and width/height
    return {
        "width": width,
        "height": height,
        "grid": grid,
        "start": start_position,
        "goal": end_position
    }

    
def get_grid(maze_data):
 # display the grid utilizing the parsed maze data 
    print(f'{maze_data['width']} (width) x {maze_data['height']} (height)')
    
    for row in maze_data['grid']:
        print(''.join(row))
'''-----------GET MAZE & GRID DATA END-----------'''



'''-----------UTILITY FUNCTIONS START-----------'''

def is_position_valid(maze_data, row, col):
    # checking for is a position is valid, boundaries of the walls 

    # check row boundary
    if row < 0 or row >= maze_data['height']: return False

    #check column boundary
    if col < 0 or col >= maze_data['width'] : return False

    # checking if the position is a wall or not 
    cell = maze_data['grid'][row][col]
    if cell == 'X':
        return False

    # otherwise, return true
    return True



def get_neighboring_positions(maze_data, position):
    #unpack to get current position
    row, col = position

    # store the valid neighbors in a empty list 
    neighbors = []

    # possible moves: up, down , left, right
    moves = [(row - 1, col), (row + 1, col), (row, col - 1), (row, col + 1)]
    # check possible moves, then append
    for new_row, new_col in moves:
        if is_position_valid(maze_data, new_row, new_col):
            neighbors.append((new_row, new_col))

    return neighbors 


def construct_path(came_from, start, goal):
    # creating an empty list, to have somewhere to store the path that was built
    path = []
    current_position = goal # working backwards from goal to start. each position will point to where i came from

    # as we work backwards from goal to start, it helps us keep track of where we came_from rather than trying to figure out how to get to.
    # seems like a lot easier impementation 
    # until 'start' is reached, the while loop will continue to run
    while current_position != start:
        # add current_position to empty list, basically logging each movement (from nodes to edges)
        path.append(current_position)
        current_position = came_from[current_position]

    # once, start is acheived, add to the 'path' list. Its technially not in came_from because we are starting from 'goal' working toward 'start'
    path.append(start)
    path.reverse()

    return path
    

'''-----------UTILITY FUNCTIONS END-----------'''





'''-----------ALGORITHM FUNCTIONS START-----------'''

def breadth_first_search(maze_data):
    # retrieveing start and goal positions from maze_data
    start = maze_data['start']
    goal = maze_data['goal']


    # BFS is queue based, need to create a queue and use the collections library
    queue = deque([start])
    visited = set([start])


    # need to track where the position came from, using a empty dictionary of lists
    came_from = {}

    # continue in the loop til queue is empty
    while queue:
        # essentially we are just iterating through the maze and if node is valid we pop it off(check it off) and proceed to the next cell/node
        current_position = queue.popleft()
        
        # need to verify if we hit the goal, build the path, if not continue until we do
        if current_position == goal:
            return construct_path(came_from, start, goal)

        # for every cell thats valid, ensure that we visited, and proceed
        for neighbor in get_neighboring_positions(maze_data, current_position):
            if neighbor not in visited:
                # https://docs.python.org/3/library/collections.html#collections.deque
                # need this link a reference to help with collections library
                came_from[neighbor] = current_position
                visited.add(neighbor)
                queue.append(neighbor)
                

    # if not path found, just return the most current
    return []
    








def depth_first_search(maze_data):
    # since we focus on exploring the farthest path possible before backtracking, this function will focus on stacking instead of queueing like BFS
    
    # retrieve the start and goal
    start = maze_data['start']
    goal = maze_data['goal']

    #stack and pop after visited
    stack = [start]
    visited = set([start])

    # keep track of where we visited, creating a dictionary of lists
    came_from = {}

    while stack:
        current_position = stack.pop()

        if current_position == goal:
            return construct_path(came_from, start, goal)

        for neighboring_cell in get_neighboring_positions(maze_data, current_position):
            if neighboring_cell not in visited: 
                visited.add(neighboring_cell)
                came_from[neighboring_cell] = current_position
                stack.append(neighboring_cell)

    # if no path not found return an empty list
    return []
    

                '''-----------GREEDY BEST FIRST SEARCH ALGORITHM & MANHATTAN DISTANCE FUNCTION-----------'''
def manhattan_distance(pos1, pos2):
    
    row1, col1 = pos1
    row2, col2 = pos2 

    distance = abs(row1-row2) + abs(col1-col2)
    return distance



# using heapq to create priority queue which enables to the function to find the best option next. Usually the lowest value
# for reference
# https://docs.python.org/3/library/heapq.html#module-heapq

def greedy_best_first_search(maze_data):

    # getting the start and goal data
    start = maze_data['start']
    goal = maze_data['goal']

    # checking for priority with heapq
    # lowest explored first
    # store in empty list, to keep track
    priority_queue = []
    heapq.heappush(priority_queue, (manhattan_distance(start, goal), start))
    '''
    manhattan to give us the value of the shortest distance and  we assign that to start, 
    because thats we essentially need to come from to get to
    '''

    # log where we visited and came_from
    visited = set([start])
    came_from = {} 

    while priority_queue: 
        current_distance, current = heapq.heappop(priority_queue) # get positon with lowest heuristic distance

        if current == goal:
            return construct_path(came_from, start, goal)

        for neighboring_cell in get_neighboring_positions(maze_data, current):
            if neighboring_cell not in visited: 
                visited.add(neighboring_cell)
                came_from[neighboring_cell] = current
            
                # calculate the hueristic distance for the neighbor
                # using the Manhattan distance function 
                heuristic_distance = manhattan_distance(neighboring_cell, goal)
                heapq.heappush(priority_queue, (heuristic_distance, neighboring_cell))

    # if not found return empty string
    return []
        
        



'''-----------ALGORITHM FUNCTIONS END-----------'''







'''-----------ALL MAZE SOLVERS START-----------'''
# function will be used to mark the path the algorithm took to get from "start" to "goal"

def mark_solution_path(maze_data, path):
    solved_grid = [row.copy() for row in maze_data["grid"]]

    for position in path:
        row, col = position # get current position 

        # using '*' to marked a visited position, verify that its not the start or goal position
        if solved_grid[row][col] not in ["S", "E"]:
            solved_grid[row][col] = "*"

    final_maze_output = f'{maze_data['width']} (width) x {maze_data['height']} (height)\n'
    for row in solved_grid:
        final_maze_output += ''.join(row) + "\n"


    # return final maze output with marked path
    return final_maze_output
        
        

''' solves for BFS'''
def maze_solver_one(maze):
    # will use maze solver one to solve BFS 
    # retrieve the maze data from the get_maze function
    maze_data = get_maze(maze)
    bfs_path = breadth_first_search(maze_data)

    # return the solution 
    if bfs_path:
        return mark_solution_path(maze_data, bfs_path)
    else:
        return maze
        
''' solves for DFS'''
def maze_solver_two(maze):
    # will use maze solver two to solve DFS
    # retrieve the maze data from the get_maze function
    maze_data = get_maze(maze)
    dfs_path = depth_first_search(maze_data)

    # return the solution 
    if dfs_path:
        return mark_solution_path(maze_data, dfs_path)
    else:
        return maze

'''solves for Greedy best-first-search'''
def maze_solver_three(maze):
    # will use maze solver two to solve gbfs
    # retrieve the maze data from the get_maze function
    maze_data = get_maze(maze)
    gbfs_path = greedy_best_first_search(maze_data)

    # return the solution 
    if gbfs_path:
        return mark_solution_path(maze_data, gbfs_path)
    else:
        return maze

'''-----------ALL MAZE SOLVERS END-----------'''        

    

Writing maze_solvers.py


In [195]:
maze_8x8 = """8 8
XXXXXXXX
X      X
X  XX  X
S  XX  E
X  XX  X
X      X
X  XX  X
XXXXXXXX"""

print("Testing 8x8 maze:")
result_bfs = maze_solver_one(maze_8x8)
result_dfs = maze_solver_two(maze_8x8)
result_gbf = maze_solver_three(maze_8x8)

print("BFS solution found:", "S" in result_bfs and "E" in result_bfs and "*" in result_bfs)
print("DFS solution found:", "S" in result_dfs and "E" in result_dfs and "*" in result_dfs)
print("GBF solution found:", "S" in result_gbf and "E" in result_gbf and "*" in result_gbf





Testing 8x8 maze:
BFS solution found: True
DFS solution found: True
GBF solution found: True


In [197]:
print("8x8 Maze Solutions with Path Lengths:")

print("\nBFS Solution:")
print(result_bfs)
bfs_path_length = result_bfs.count('*') + 2  # +2 for S and E
print(f"BFS path length: {bfs_path_length} positions")

print("\nDFS Solution:")  
print(result_dfs)
dfs_path_length = result_dfs.count('*') + 2  # +2 for S and E
print(f"DFS path length: {dfs_path_length} positions")

print("\nGreedy Best-First Solution:")
print(result_gbf)
gbf_path_length = result_gbf.count('*') + 2  # +2 for S and E
print(f"GBF path length: {gbf_path_length} positions")

print("\nComparison:")
print(f"BFS: {bfs_path_length} positions")
print(f"DFS: {dfs_path_length} positions") 
print(f"GBF: {gbf_path_length} positions")

8x8 Maze Solutions with Path Lengths:

BFS Solution:
8 (width) x 8 (height)
XXXXXXXX
X***** X
X* XX* X
S* XX**E
X  XX  X
X      X
X  XX  X
XXXXXXXX

BFS path length: 12 positions

DFS Solution:
8 (width) x 8 (height)
XXXXXXXX
X      X
X  XX  X
S**XX *E
X *XX *X
X *****X
X  XX  X
XXXXXXXX

DFS path length: 12 positions

Greedy Best-First Solution:
8 (width) x 8 (height)
XXXXXXXX
X *****X
X *XX *X
S**XX *E
X  XX  X
X      X
X  XX  X
XXXXXXXX

GBF path length: 12 positions

Comparison:
BFS: 12 positions
DFS: 12 positions
GBF: 12 positions


In [198]:
maze_8x15 = """8 15
XXXXXXXX
X      X
X  XX  X
S      X
X  XX  X
X      X
X  XX  X
X      X
X  XX  X
X      X
X  XX  X
X      X
X  XX  X
X      E
XXXXXXXX"""

print("Testing 8x15 maze:")
result_bfs_8x15 = maze_solver_one(maze_8x15)
result_dfs_8x15 = maze_solver_two(maze_8x15)  
result_gbf_8x15 = maze_solver_three(maze_8x15)

print("\n8x15 Maze Solutions with Path Lengths:")

print("\nBFS Solution:")
print(result_bfs_8x15)
bfs_path_length = result_bfs_8x15.count('*') + 2
print(f"BFS path length: {bfs_path_length} positions")

print("\nDFS Solution:")
print(result_dfs_8x15)
dfs_path_length = result_dfs_8x15.count('*') + 2
print(f"DFS path length: {dfs_path_length} positions")

print("\nGreedy Best-First Solution:")
print(result_gbf_8x15)
gbf_path_length = result_gbf_8x15.count('*') + 2
print(f"GBF path length: {gbf_path_length} positions")

print("\nComparison:")
print(f"BFS: {bfs_path_length} positions")
print(f"DFS: {dfs_path_length} positions")
print(f"GBF: {gbf_path_length} positions")

Testing 8x15 maze:

8x15 Maze Solutions with Path Lengths:

BFS Solution:
8 (width) x 15 (height)
XXXXXXXX
X      X
X  XX  X
S*     X
X* XX  X
X*     X
X* XX  X
X*     X
X* XX  X
X*     X
X* XX  X
X*     X
X* XX  X
X******E
XXXXXXXX

BFS path length: 18 positions

DFS Solution:
8 (width) x 15 (height)
XXXXXXXX
X      X
X  XX  X
S******X
X  XX *X
X******X
X* XX  X
X******X
X  XX *X
X******X
X* XX  X
X******X
X  XX *X
X     *E
XXXXXXXX

DFS path length: 38 positions

Greedy Best-First Solution:
8 (width) x 15 (height)
XXXXXXXX
X      X
X  XX  X
S******X
X  XX *X
X     *X
X  XX *X
X     *X
X  XX *X
X     *X
X  XX *X
X     *X
X  XX *X
X     *E
XXXXXXXX

GBF path length: 18 positions

Comparison:
BFS: 18 positions
DFS: 38 positions
GBF: 18 positions


In [199]:
# This maze has multiple branching paths with dead ends to showcase algorithm differences
maze_showcase = """12 10
XXXXXXXXXXXX
S          X
XXXXXXXXX  X
X       X  X
X XXXXX X  X
X X   X X  X
X X X X X  X
X   X   X  X
XXXXXXXXX  X
X        E X
XXXXXXXXXXXX"""

print("Testing showcase maze (designed to show algorithm differences):")
import time

# Test with timing
start = time.time()
result_bfs_show = maze_solver_one(maze_showcase)
bfs_time = time.time() - start

start = time.time()
result_dfs_show = maze_solver_two(maze_showcase)
dfs_time = time.time() - start

start = time.time()
result_gbf_show = maze_solver_three(maze_showcase)
gbf_time = time.time() - start

print(f"\nBFS: {result_bfs_show.count('*') + 2} positions in {bfs_time:.4f}s")
print(f"DFS: {result_dfs_show.count('*') + 2} positions in {dfs_time:.4f}s")
print(f"GBF: {result_gbf_show.count('*') + 2} positions in {gbf_time:.4f}s")

# Show the paths
print("\nBFS Solution:")
print(result_bfs_show)
print("\nDFS Solution:")
print(result_dfs_show)
print("\nGBF Solution:")
print(result_gbf_show)

Testing showcase maze (designed to show algorithm differences):

BFS: 18 positions in 0.0002s
DFS: 22 positions in 0.0001s
GBF: 18 positions in 0.0006s

BFS Solution:
12 (width) x 10 (height)
XXXXXXXXXXXX
S********* X
XXXXXXXXX* X
X       X* X
X XXXXX X* X
X X   X X* X
X X X X X* X
X   X   X* X
XXXXXXXXX* X
X        E X
XXXXXXXXXXXX


DFS Solution:
12 (width) x 10 (height)
XXXXXXXXXXXX
S**********X
XXXXXXXXX *X
X       X**X
X XXXXX X* X
X X   X X**X
X X X X X *X
X   X   X**X
XXXXXXXXX* X
X        E X
XXXXXXXXXXXX


GBF Solution:
12 (width) x 10 (height)
XXXXXXXXXXXX
S********* X
XXXXXXXXX* X
X       X* X
X XXXXX X* X
X X   X X* X
X X X X X* X
X   X   X* X
XXXXXXXXX* X
X        E X
XXXXXXXXXXXX



In [200]:
# Generate a simple 50x50 maze programmatically
def create_test_maze(width, height):
    maze_lines = [f"{width} {height}"]
    
    for row in range(height):
        if row == 0 or row == height - 1:
            # Top and bottom walls
            maze_lines.append("X" * width)
        elif row == 1:
            # Second row with start
            maze_lines.append("X" + "S" + " " * (width-4) + "X")
        elif row == height - 2:
            # Second to last row with end
            maze_lines.append("X" + " " * (width-4) + "E" + "X")
        else:
            # Middle rows - walls on sides, open in middle
            maze_lines.append("X" + " " * (width-2) + "X")
    
    return "\n".join(maze_lines)

maze_50x50 = create_test_maze(50, 50)

print("Testing 50x50 maze:")
import time

start_time = time.time()
result_bfs_50 = maze_solver_one(maze_50x50)
bfs_time = time.time() - start_time

start_time = time.time()
result_dfs_50 = maze_solver_two(maze_50x50)
dfs_time = time.time() - start_time

start_time = time.time()
result_gbf_50 = maze_solver_three(maze_50x50)
gbf_time = time.time() - start_time

print(f"BFS time: {bfs_time:.4f}s")
print(f"DFS time: {dfs_time:.4f}s") 
print(f"GBF time: {gbf_time:.4f}s")

Testing 50x50 maze:
BFS time: 0.0047s
DFS time: 0.0023s
GBF time: 0.0008s


In [201]:
maze_1000x1000 = create_test_maze(1000, 1000)

print("Testing 1000x1000 maze (performance test):")
print("Warning: This may take several minutes...")

# Test just one algorithm first to check performance
start_time = time.time()
result_bfs_1000 = maze_solver_one(maze_1000x1000)
bfs_time_1000 = time.time() - start_time

print(f"BFS completed in: {bfs_time_1000:.2f}s")
print(f"Solution found: {'*' in result_bfs_1000}")

# Only test other algorithms if BFS completes reasonably quickly
if bfs_time_1000 < 60:  # Less than 1 minute
    print("Testing other algorithms...")
    # Test DFS and GBF
else:
    print("Skipping other algorithms due to performance concerns")

Testing 1000x1000 maze (performance test):
BFS completed in: 2.51s
Solution found: True
Testing other algorithms...
